# 07 Property Search with LLM Filter Extraction

Use this notebook to load the sample listings dataset, test exact property filters, and try natural-language property search parsing with Ollama.

In [1]:
import os
import sys

sys.path.append(os.path.abspath('..'))

In [2]:
from src.api.schemas import PropertySearchFilters
from src.data.load_property_listings import main as load_property_listings_main
from src.db.repository import search_property_listings
from src.db.session import SessionLocal, create_db_tables
from src.search.parser import parse_property_search_query

In [3]:
create_db_tables()
load_property_listings_main()

Loaded 15 property listings from /Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/data/sample/property_listings_seed.csv


In [4]:
db = SessionLocal()
filters = PropertySearchFilters(city='San Jose', max_price_usd=900000, limit=5)
records = search_property_listings(db, filters)
[(record.listing_code, record.city, record.asking_price_usd) for record in records]

[('CA-SJ-002', 'San Jose', 785000.0)]

In [5]:
parsed_filters, parser_model_name = parse_property_search_query(
    'Find me a 2 bedroom condo in Oakland under 800000',
    limit=5,
)
parser_model_name, parsed_filters

('qwen2.5-coder:7b-instruct',
 PropertySearchFilters(city='Oakland', locality=None, property_type='condo', min_price_usd=None, max_price_usd=800000.0, min_bedrooms=2, max_bedrooms=2, min_bathrooms=None, max_bathrooms=None, min_area_sqft=None, max_area_sqft=None, limit=5, sort_by='asking_price_usd', sort_order='asc'))

In [6]:
records = search_property_listings(db, parsed_filters)
[(record.listing_code, record.title, record.asking_price_usd) for record in records]

[('CA-OAK-001', 'Lake Merritt Condo', 725000.0)]

In [7]:
db.close()